# 02 - Random Forest Baseline

**Purpose:** a classical machine-learning baseline using interpretable global
colour and texture features.

Before training any deep network we establish how far a simple model can get
from hand-crafted descriptors of each photo (RGB/HSV colour statistics and
histograms, plus an LBP texture histogram). This baseline sets the bar that the
later CNN models must beat, and it is fully transparent: every feature and every
post-processing step is visible in this notebook.


## 1. Imports and setup


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

from src import config, data, targets, splits, metrics, plots, features

RANDOM_SEED = config.RANDOM_SEED
config.ensure_output_dirs()
print("Project root:", ROOT)


## 2-5. Load metadata, build targets, and validate

We load the metadata, build the one-row-per-image table and the Code-level target vectors, then validate both before modelling.


In [ ]:
metadata = data.load_metadata()
image_df = data.build_image_dataframe(metadata)

vocabulary = targets.build_material_vocabulary(metadata)
target_vectors = targets.build_targets(metadata, vocabulary=vocabulary)
materials = target_vectors["materials"]

# Validate the dataset and the target vectors before training.
report = data.validate_dataset(metadata)
assert report["ok"], report["problems"]
targets.validate_targets(target_vectors)

print(f"{len(metadata)} mixtures, {len(image_df)} images")
print("materials:", materials)


## 6. Grouped train / validation / test split by Code

The five photos of one `Code` show the **same physical mixture**. Splitting at
the image level would let near-duplicate photos of one mixture appear in both
train and test, leaking the answer. So we split by `Code`: all five images of a
mixture always stay in the same split. The assertions below fail loudly if that
guarantee is ever broken.


In [ ]:
train_df, val_df, test_df = splits.grouped_train_val_test_split(
    image_df, group_col="Code",
    test_size=config.TEST_SIZE, val_size=config.VAL_SIZE, seed=RANDOM_SEED,
)

splits.check_no_leakage(train_df, val_df, test_df)
splits.check_groups_intact([train_df, val_df, test_df], image_df)

print("images -> train {}, val {}, test {}".format(
    len(train_df), len(val_df), len(test_df)))
print("codes  -> train {}, val {}, test {}".format(
    train_df["Code"].nunique(), val_df["Code"].nunique(), test_df["Code"].nunique()))


## 7. Extract hand-crafted image features

For every image we compute interpretable colour statistics, colour histograms,
and an LBP texture histogram (see `src/features.py`). This decodes all 805
photos, so expect it to take a few minutes.


In [ ]:
feature_df = features.extract_features_for_dataframe(
    image_df, image_size=config.IMAGE_SIZE, hist_bins=16, include_texture=True,
)
feature_cols = [c for c in feature_df.columns if c not in ("Code", "image_path")]

# The feature matrix must be clean before it reaches the model.
X_all = feature_df[feature_cols].to_numpy(dtype=float)
assert np.isfinite(X_all).all(), "feature matrix contains NaN or inf"

print(f"feature matrix: {X_all.shape[0]} images x {len(feature_cols)} features")
print("first features:", feature_cols[:6])
print("last features :", feature_cols[-3:])


## 8. Prepare the target matrix

The baseline target is the **mass fraction**. It is derived directly from the
measured weights, which makes it the most interpretable first target for a
classical regression baseline. The Code-level target vector is shared by all
five images of that Code.


In [ ]:
mass_by_code = {code: target_vectors["mass_fraction"][i]
                for i, code in enumerate(target_vectors["codes"])}
feat_by_path = feature_df.set_index("image_path")

def make_xy(split_df):
    """Feature matrix X and target matrix y for one split, in row order."""
    X = feat_by_path.loc[split_df["image_path"], feature_cols].to_numpy(dtype=float)
    y = np.array([mass_by_code[c] for c in split_df["Code"]])
    return X, y

X_train, y_train = make_xy(train_df)
X_val, y_val = make_xy(val_df)
X_test, y_test = make_xy(test_df)
print("X_train", X_train.shape, "| y_train", y_train.shape)


## 9. Train the Random Forest

A single multi-output `RandomForestRegressor` predicts all six material fractions at once. Reproducible via `RANDOM_SEED`.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    min_samples_leaf=2,
)
rf.fit(X_train, y_train)
print("trained:", rf.n_estimators, "trees, multi-output ->", y_train.shape[1], "materials")


## 10. Predictions as valid composition vectors

A Random Forest has no notion of a composition, so its raw outputs need not be
non-negative or sum to one. We turn each raw output into a valid composition:
**clip** negatives to 0, fall back to a **uniform** vector if a row is all zeros,
then **renormalize** every row to sum to 1.


In [ ]:
def to_composition(raw):
    """Clip negatives, use a uniform vector for all-zero rows, renormalize to 1."""
    p = np.clip(np.asarray(raw, dtype=float), 0.0, None)
    row_sums = p.sum(axis=1, keepdims=True)
    p = np.where(row_sums == 0, 1.0, p)              # all-zero rows -> uniform
    return p / p.sum(axis=1, keepdims=True)

raw_pred_train, raw_pred_val, raw_pred_test = (
    rf.predict(X_train), rf.predict(X_val), rf.predict(X_test))
pred_train = to_composition(raw_pred_train)
pred_val = to_composition(raw_pred_val)
pred_test = to_composition(raw_pred_test)

# Sanity: every predicted row is a valid composition.
assert (pred_test >= 0).all()
assert np.allclose(pred_test.sum(axis=1), 1.0)
print("predictions are valid compositions (non-negative, rows sum to 1)")


## 11. Evaluation (image level)

Every model in this thesis is scored with the same metrics from `src.metrics`, so the numbers are directly comparable.


In [ ]:
image_metrics = {
    "train": metrics.evaluate_composition(y_train, pred_train, materials),
    "val":   metrics.evaluate_composition(y_val,   pred_val,   materials),
    "test":  metrics.evaluate_composition(y_test,  pred_test,  materials),
}
image_metrics_df = pd.DataFrame(image_metrics).T
image_metrics_df.index.name = "split"
image_metrics_df[["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]].round(4)


## 12. Mixture-level evaluation

The final object of interest is the **physical mixture**, not the individual
photo. So for each test Code we average its five image-level predictions and
renormalize, then score against the Code-level target. We report the test set
**both** ways: image-level and mixture-level.


In [ ]:
def to_mixture_level(split_df, pred):
    """Average the per-image predictions within each Code, then renormalize."""
    df = pd.DataFrame(pred, columns=materials)
    df["Code"] = split_df["Code"].values
    avg = df.groupby("Code")[materials].mean()
    avg = avg.div(avg.sum(axis=1), axis=0)           # renormalize to sum to 1
    codes = avg.index.tolist()
    y_true = np.array([mass_by_code[c] for c in codes])
    return codes, y_true, avg.to_numpy()

test_codes_mix, y_test_mix, pred_test_mix = to_mixture_level(test_df, pred_test)
mixture_metrics = metrics.evaluate_composition(y_test_mix, pred_test_mix, materials)

compare_df = pd.DataFrame({
    "test_image_level": image_metrics["test"],
    "test_mixture_level": mixture_metrics,
}).T[["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]].round(4)
compare_df


## 13. Save predictions and metrics


In [ ]:
# Metrics CSV: train / val / test (image level) plus test (mixture level).
all_metrics = dict(image_metrics)
all_metrics["test_mixture"] = mixture_metrics
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "split"
metrics_path = config.TABLE_DIR / "baseline_random_forest_metrics.csv"
metrics_df.to_csv(metrics_path)

# Predictions CSV: one row per image, with true / predicted / raw columns.
def predictions_frame(split_df, y_true, raw_pred, proc_pred, split_name):
    out = pd.DataFrame({
        "Code": split_df["Code"].values,
        "image_path": split_df["image_path"].values,
        "split": split_name,
    })
    for j, m in enumerate(materials):
        out[f"true_{m}"] = y_true[:, j]
        out[f"pred_{m}"] = proc_pred[:, j]
        out[f"raw_pred_{m}"] = raw_pred[:, j]
    return out

pred_csv = pd.concat([
    predictions_frame(train_df, y_train, raw_pred_train, pred_train, "train"),
    predictions_frame(val_df,   y_val,   raw_pred_val,   pred_val,   "val"),
    predictions_frame(test_df,  y_test,  raw_pred_test,  pred_test,  "test"),
], ignore_index=True)
pred_path = config.PREDICTION_DIR / "baseline_random_forest_predictions.csv"
pred_csv.to_csv(pred_path, index=False)

print("saved:", metrics_path.name, "and", pred_path.name)


## 14. Plots

Predicted-vs-true scatter and per-material MAE on the test set, plus example predictions for a few test mixtures.


In [ ]:
plots.plot_predicted_vs_true(y_test, pred_test, materials,
                             save_path="02_pred_vs_true_test.png")
plt.show()

test_per_material = metrics.per_material_mae(y_test, pred_test, materials)
plots.plot_per_material_mae(test_per_material, save_path="02_per_material_mae_test.png")
plt.show()

# Example predictions on a few test mixtures (mixture-level, averaged over 5 images).
example_codes = test_codes_mix[:4]
plots.plot_example_predictions(
    image_df, example_codes, y_test_mix[:4], pred_test_mix[:4], materials,
    save_path="02_example_predictions_test.png",
)
plt.show()


## 15. Interpretation

**What the baseline does.** It describes each photo with global colour
statistics, colour histograms and an LBP texture histogram, then a Random Forest
regresses the six material mass fractions. The raw outputs are clipped and
renormalized into a valid composition.

**Why grouped splitting.** All five photos of a mixture share one composition,
so the split is grouped by `Code`. This prevents the model from "recognizing" a
mixture it already saw in training and gives an honest estimate of how it
generalizes to *new* mixtures.

**Reading the metrics.** `MAE`/`RMSE` are the average error on each predicted
material fraction (lower is better); `R²` is how much of the composition
variance is explained (1.0 is perfect, 0.0 is no better than predicting the
mean). Compare the image-level and mixture-level rows above: averaging the five
photos of a mixture typically *reduces* the error, which is why the physical
mixture is the right unit of evaluation.

**Mass leakage onto absent materials.** `false_positive_absent_%` is the share
of predicted mass placed on materials that are actually absent. A Random Forest
rarely predicts an exact zero, so some mass always leaks onto absent materials.
This is exactly the weakness the later sparse multitask model (notebook 04) is
designed to reduce.

**Why this baseline matters.** It is simple, fast and fully interpretable, and
it quantifies how much signal lives in plain colour/texture statistics. The deep
models in notebooks 03-04 are only worthwhile if they clearly beat this bar.
